In [32]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder,OneHotEncoder,TargetEncoder

In [33]:
df=pd.read_csv("AmesHousing - AmesHousing.csv.csv")
df=df.drop(columns=["PID","Order"])
df["MS SubClass"] = df["MS SubClass"].astype(str)

In [34]:
train=df[df["Yr Sold"]<=2008]
val=df[df["Yr Sold"]==2009]
test=df[df["Yr Sold"]==2010]

In [35]:
target="SalePrice"
X_train, y_train = train.drop(columns=[target]), train[target]
X_val,   y_val   = val.drop(columns=[target]), val[target]
X_test,  y_test  = test.drop(columns=[target]), test[target]

In [36]:
print(len(X_train), len(X_val), len(X_test))

1941 648 341


In [37]:
qual_cols = ["Exter Qual", "Kitchen Qual", "Bsmt Qual", "Heating QC"]
lot_cols = ["Lot Shape"]


In [38]:
qual_order = ["None", "Po", "Fa", "TA", "Gd", "Ex"]
lot_order = ["IR3", "IR2", "IR1", "Reg"]

In [70]:
ordinal_encoder = OrdinalEncoder(
    categories=[qual_order] * 4 + [lot_order],
    #categories=[qual_order, qual_order,  qual_order, qual_order,  lot_order]
    #           (Exter Qual  Kitchen Qual Bsmt Qual   Heating QC   Lot Shape)
    handle_unknown="use_encoded_value",
    unknown_value=-1,
).set_output(transform="pandas")

In [40]:
ord_cols = qual_cols + lot_cols

X_train_ord = ordinal_encoder.fit_transform(X_train[ord_cols].fillna("None"))
X_val_ord   = ordinal_encoder.transform(X_val[ord_cols].fillna("None"))
X_test_ord  = ordinal_encoder.transform(X_test[ord_cols].fillna("None"))

In [41]:
# Kitchen Qual = "Po" chỉ có ở Test -> kỳ vọng mã 1, không phải -1
print(X_test_ord.loc[X_test["Kitchen Qual"] == "Po", "Kitchen Qual"].tolist())

[1.0]


In [42]:
onehot_cols = ["MS Zoning", "House Style", "Foundation", "Garage Type",
               "Sale Condition", "Central Air", "Utilities"]

In [43]:
onehot_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False).set_output(transform="pandas")

In [59]:
X_train_one_hot = onehot_encoder.fit_transform(X_train[onehot_cols].fillna("None"))
X_val_one_hot  = onehot_encoder.transform(X_val[onehot_cols].fillna("None"))
X_test_one_hot  = onehot_encoder.transform(X_test[onehot_cols].fillna("None"))

In [61]:
# Validation có Utilities = "NoSeWa" chưa thấy trong Train
print("Utilities Train:", X_train["Utilities"].unique(), "| Val:", X_val["Utilities"].unique())
X_val_one_hot.loc[X_val["Utilities"] == "NoSeWa", [c for c in X_val_one_hot.columns if c.startswith("Utilities")]]

Utilities Train: <ArrowStringArray>
['AllPub', 'NoSewr']
Length: 2, dtype: str | Val: <ArrowStringArray>
['AllPub', 'NoSewr', 'NoSeWa']
Length: 3, dtype: str


,Utilities_AllPub,Utilities_NoSewr
955,0.0,0.0


In [46]:
high_cols = ["Neighborhood", "MS SubClass"]
print(X_train[high_cols].nunique())#Độ đa dạng giá trị của một cột hạng mục 

Neighborhood    28
MS SubClass     16
dtype: int64


In [47]:
# Target Encoding: Train -> fit_transform (giá trị Out-of-Fold); Val/Test -> transform
target_encoder = TargetEncoder(cv=5, 
                               smooth="auto", 
                               target_type="continuous", 
                               random_state=42).set_output(transform="pandas")

In [48]:
X_train_te = target_encoder.fit_transform(X_train[high_cols], y_train).add_suffix("_te")
X_val_te   = target_encoder.transform(X_val[high_cols]).add_suffix("_te")
X_test_te  = target_encoder.transform(X_test[high_cols]).add_suffix("_te")

In [49]:
# Count Encoding: số lần xuất hiện chỉ tính trên Train, danh mục lạ -> 0
X_train_cnt, X_val_cnt, X_test_cnt = [
    pd.DataFrame({c + "_count": X[c].map(X_train[c].value_counts()).fillna(0) for c in high_cols})
    for X in (X_train, X_val, X_test)
]

In [50]:
# So sánh khu vực hiếm (Landmrk, GrnHill) với khu vực phổ biến (NAmes)
names = ["Landmrk", "GrnHill", "NAmes"]
full_map = pd.Series(target_encoder.encodings_[0], index=target_encoder.categories_[0])  # bảng học từ toàn bộ Train

In [51]:
pd.DataFrame({
    "Số mẫu Train": X_train["Neighborhood"].value_counts().reindex(names),
    "Giá TB thực": y_train.groupby(X_train["Neighborhood"]).mean().reindex(names),
    "Bảng mã hóa toàn Train (dùng cho Val/Test)": full_map.reindex(names),
    "OOF trung bình (Train)": X_train_te["Neighborhood_te"].groupby(X_train["Neighborhood"]).mean().reindex(names),
}).round(0)

,Số mẫu Train,Giá TB thực,Bảng mã hóa toàn Train (dùng cho Val/Test),OOF trung bình (Train)
Landmrk,1,137000.0,137000.0,181740.0
GrnHill,2,280000.0,264121.0,280000.0
NAmes,290,144507.0,144526.0,144530.0


In [52]:
def add_month(X):
    return pd.DataFrame({
        "Mo Sold_sin": np.sin(2 * np.pi * X["Mo Sold"] / 12),
        "Mo Sold_cos": np.cos(2 * np.pi * X["Mo Sold"] / 12),
    })

In [53]:
X_train_m, X_val_m, X_test_m = add_month(X_train), add_month(X_val), add_month(X_test)

In [54]:
m = pd.DataFrame({"Mo Sold": [12, 1, 2]})
p = add_month(m).values
print("Khoảng cách 12-1:", np.linalg.norm(p[0] - p[1]).round(3), "| 1-2:", np.linalg.norm(p[1] - p[2]).round(3))

Khoảng cách 12-1: 0.518 | 1-2: 0.518


In [62]:
X_train_ready = pd.concat([X_train_ord, X_train_one_hot, X_train_te, X_train_cnt, X_train_m], axis=1)
X_val_ready   = pd.concat([X_val_ord,   X_val_one_hot,   X_val_te,   X_val_cnt,   X_val_m],   axis=1)
X_test_ready  = pd.concat([X_test_ord,  X_test_one_hot,  X_test_te,  X_test_cnt,  X_test_m],  axis=1)

In [63]:
sets = {"Train": X_train_ready, "Validation": X_val_ready, "Test": X_test_ready}

In [64]:
pd.DataFrame({
    "Số dòng": [len(d) for d in sets.values()],
    "Số cột": [d.shape[1] for d in sets.values()],
    "Tổng NaN": [int(d.isna().sum().sum()) for d in sets.values()],
    "Toàn cột số": [all(pd.api.types.is_numeric_dtype(t) for t in d.dtypes) for d in sets.values()],
    "Cột giống Train": [list(d.columns) == list(X_train_ready.columns) for d in sets.values()],
}, index=sets.keys())

,Số dòng,Số cột,Tổng NaN,Toàn cột số,Cột giống Train
Train,1941,49,0,True,True
Validation,648,49,0,True,True
Test,341,49,0,True,True


In [65]:
X_train_ready.to_csv("X_train_ready.csv", index=False)
X_val_ready.to_csv("X_val_ready.csv", index=False)
X_test_ready.to_csv("X_test_ready.csv", index=False)
y_train.to_csv("y_train.csv", index=False)
y_val.to_csv("y_val.csv", index=False)
y_test.to_csv("y_test.csv", index=False)